# 第 1 周末练习 —— LLM 提示器（Prompt Engineer 助手）

## 练习目标（理念）

为了展示你对 **OpenAI API**（经 OpenRouter）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术 / 科学问题（本例：为什么不能直接从月球取太阳能？）
- **输出（GPT 路径）**：由「资深提示工程师」角色生成结构化 JSON prompt，并附带评分与犀利点评
- **输出（Llama 路径）**：用一份已写好的结构化 prompt 向本地 `llama3.2` 要详细解释
- **额外要求**：GPT 路径用**流式（streaming）**边生成边刷新 Markdown

这是课程期间也能天天用的「提示词工坊」：先让云端模型帮你写更好的 prompt，再丢给本地模型作答。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么写 prompt」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接 `delta.content`，`update_display` 刷新 |
| 兼容网关 OpenRouter | `base_url=https://openrouter.ai/api/v1` |
| Ollama 本地模型 | `llama3.2`，OpenAI 兼容口 `http://localhost:11434/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPEN_ROUTER_API_KEY`（注意本练习用的名字不是 `OPENAI_API_KEY`）
3. 若要跑 Llama：本机先 `ollama pull llama3.2`，并保证 Ollama 在 `localhost:11434` 监听
4. 在提问单元格改写 `question`，对比 GPT（写 prompt）与 Llama（按 prompt 答题）两条路径


In [35]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染 + 流式 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：同时用于 OpenRouter 与本地 Ollama（兼容 /v1）
from openai import OpenAI


In [42]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenRouter 上选用的云端小模型（字符串必须是网关认识的 model id）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'


In [37]:
# ========== 环境：加载 OpenRouter 密钥并做形态检查 ==========

# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 注意变量名是 OPEN_ROUTER_API_KEY（带下划线），不是常见的 OPENAI_API_KEY
openRouter_api_key = os.getenv("OPEN_ROUTER_API_KEY")
# OpenRouter 的 OpenAI 兼容基址（改错会导致请求打到错误主机）
openRouter_url = "https://openrouter.ai/api/v1"

# 分层检查钥匙：缺失 / 前缀不对 / 首尾空白 —— 提示文案保持英文原样
if not openRouter_api_key:
    print("No open router API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not openRouter_api_key.startswith("sk-"):
    print("An open router API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook")
elif openRouter_api_key.strip() != openRouter_api_key:
    print("An open router API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("Open router API key found and looks good so far!")


Open router API key found and looks good so far!


In [38]:
# ========== 提问 + system prompt：让 GPT 扮演「资深提示工程师」==========
# 下面所有 prompt 字符串保持英文：改译会改变模型角色与输出结构

# JSON 模板骨架：告诉模型「写出来的 prompt 应长什么样」（字段先留空，由模型填充）
prompt_template = """
{
  "system_prompt": "",
  "user_prompt": "",
  "language": "",
  "format": "",
  "instructions": {
    "response_length": "",
    "language_support": "",
    "clarification_prompt": ""
  }
}

"""
# system：角色设定 + 要求输出 JSON 结构 prompt + 评分/犀利点评
system_prompt = f"""
    You are a senior prompt engineer from anthropic. 
    You have a deep understanding of the prompt engineering and the ability to write prompts that get the most out of the model. 
    Your role is to write a prompt that will get the best answer from each question that the user asks.
    Your prompt should be able to handle the question in any language and in any format.
    You prompt should be in json format and must clearly define a system prompt and a user prompt, with other instructions as needed.

    Along with the prompt, you should also provide a short explanation of the prompt and the rationale behind it, and score it out of 10.
    Provide a brutaly honest critique of the prompt and suggest improvements.

    Structure of the prompt should be as follows: {prompt_template}
    """

# 用户真正要问的问题：改这里即可换题（内容保持原样时可直接跑）
question = """
    Why cant we get the solar energy straight from the moon?
"""


In [ ]:
# ========== GPT 路径：经 OpenRouter 流式调用 gpt-4o-mini ==========

# messages：system 定「怎么写 prompt」，user 放具体问题
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# 客户端指向 OpenRouter 基址；密钥用上面读到的 openRouter_api_key
openai = OpenAI(base_url=openRouter_url,api_key=openRouter_api_key)
# stream=True：边生成边返回增量 delta，不用等整段结束
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
# 累积完整回复文本，供每次刷新 Markdown
response = ""

# 先占一个可更新的 Markdown 显示位（display_id=True）
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk：把新文本拼上去并刷新同一显示位
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# 下一行是 IPython shell magic：拉取本地 llama3.2（需已安装并启动 Ollama）
!ollama pull llama3.2


In [41]:
# ========== Ollama 客户端：走 OpenAI 兼容的 /v1 接口 ==========

# 本地兼容基址（不是原生 /api/chat）；需 Ollama 正在运行
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；习惯写 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== Llama 路径：用「已写好的结构化 prompt」向本地模型要详细解释 ==========
# 注意：这里的 prompt 字典 / system_content 字符串都是发给模型的指令，保持英文原样

# 一份完整 JSON 风格配置：system / user / 语言 / 格式 / 长度等（可对照上格 GPT 生成结果手工微调）
prompt = {
    "system_prompt": "You are a highly knowledgeable assistant capable of providing detailed explanations on scientific concepts. Your goal is to give clear, accurate, and insightful answers to the user's questions.",
    "user_prompt": "Why can't we get the solar energy straight from the moon? Please explain in detail, considering scientific principles, technology, and any relevant challenges.",
    "language": "English",
    "format": "detailed explanation with sections",
    "instructions": {
        "response_length": "comprehensive, 300-500 words",
        "language_support": "respond in the same language as the user's question",
        "clarification_prompt": "If the question is ambiguous, ask for clarification before answering."
    }
}

# 把字典字段展开进真正的 system 消息，让模型同时看到角色与格式约束
system_content = f"""{prompt['system_prompt']}

Response language: {prompt['language']}
Response format: {prompt['format']}
Response length: {prompt['instructions']['response_length']}
Language support: {prompt['instructions']['language_support']}
Clarification: {prompt['instructions']['clarification_prompt']}
"""

# messages：system 放展开后的约束，user 放具体提问
messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": prompt["user_prompt"]}
]


# 非流式：等整段生成完再取 content（与上格 GPT 流式对比）
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
answer = response.choices[0].message.content

# 在笔记本里用 Markdown 漂亮展示最终答案
display(Markdown(answer))
